In [ ]:
#Dataset to learn/unlearn
from sklearn.datasets import fetch_20newsgroups
#Vectoriser to convert text in the files
from sklearn.feature_extraction.text import TfidfVectorizer as TIV
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

#For Fine-Tuning-based Unlearning
from copy import deepcopy

#For NegGrad
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

#Used for MIA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import numpy
import random

#For Web Application Display
import mercury as mr
import pandas as pd
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def logLoss(targets, predictProbs):
    sum = 0
    #For every class
    for i in range(20):
        currTargets = []
        for j in targets:
            if j == i:
                currTargets.append(1)
            else:
                currTargets.append(0)
        sum = sum + metrics.log_loss(currTargets, predictProbs[:,i], labels=[0,1])
    sum = sum / 20
    return sum

def getMatches(email):
    newsgroupsEx = fetch_20newsgroups(subset='train', remove=('footers', 'quotes'), random_state=randomStan)
    count = []
    for i in range(len(newsgroupsEx.data)):
        if newsgroupsEx.data[i].__contains__(email):
            count.append(i)
    return count

def getExample():
    exRan = random.randrange(2048)
    newsgroupsEx = fetch_20newsgroups(subset='test', remove=('footers', 'quotes'), random_state=exRan)
    newsgroupsProper = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), random_state=exRan)
    return newsgroupsEx.data[0], newsgroupsProper.data[0], newsgroupsProper.target[0]
                                      

def miaBuilder(numShadows, predTrain, predTest):
    print("Training Shadow Models...", end='\r')
    predictions = []
    labels = []
    #Generate shadow models for the MIA/Attacker model to use for training
    for i in range(numShadows):
        print("Training Shadow", i+1, "of", numShadows, end='\r')
        randomShade = random.randrange(2048)

        #Prepare shadow models
        shadowData = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'), random_state=randomShade)

        shadowTrainData, shadowTestData, shadowTrainTarget, _ = train_test_split(
        shadowData.data, shadowData.target, train_size=0.5, stratify=shadowData.target, random_state=42
                                                                )

        shadowVec = TIV(stop_words=stopWords, max_features=maxFeatures)
        shadowVecTrain = shadowVec.fit_transform(shadowTrainData)
        shadowVecTest = shadowVec.transform(shadowTestData)

        shadowClassifier = MultinomialNB(alpha=0.1)
        shadowClassifier.fit(shadowVecTrain, shadowTrainTarget)

        #Get predictions for both training and test data
        shadowPredTrain = shadowClassifier.predict_log_proba(shadowVecTrain)
        shadowPredTest = shadowClassifier.predict_log_proba(shadowVecTest)

        #Mark with member data (training data: ones), and non-member data (test data: zeros)
        shadowLabelTrain = numpy.ones(len(shadowPredTrain))
        shadowLabelTest = numpy.zeros(len(shadowPredTest))

        #Combine results of the current shadow model, and append to the larger set of results
        predictions.append(numpy.concatenate((shadowPredTrain, shadowPredTest)))
        labels.append(numpy.concatenate((shadowLabelTrain, shadowLabelTest)))
    
    print("Shadow Models Training Complete!")
    #Combine the results of the shadow models
    miaOut = numpy.concatenate(predictions)
    miaLabels = numpy.concatenate(labels)

    #Scale the outputs
    global scaler
    miaOutScaled = scaler.fit_transform(miaOut)

    #Split data for training attack model
    trainOut, testOut, trainLabels, testLabels = train_test_split(
        miaOutScaled, miaLabels, test_size=0.3, stratify=miaLabels, random_state=random.randrange(2048)
                                                                )
    
    #Train MIA
    print("Training MIA Model...", end='\r')
    miaAttacker = LogisticRegression(solver='liblinear', random_state=random.randrange(2048))
    miaAttacker.fit(trainOut, trainLabels)

    #Evaluate on test data
    predictions = miaAttacker.predict(testOut)

    #Use predTrain and predTest
    attackerTest = numpy.concatenate((predTrain, predTest))
    attackerLabels = numpy.concatenate([numpy.ones(len(predTrain)), numpy.zeros(len(predTest))])

    #Scale using scaler from shadow model data
    attackerTotalScaled = scaler.transform(attackerTest)
    attackerTrainScaled = scaler.transform(predTrain)
    attackerTestScaled = scaler.transform(predTest)

    #Make predictions
    miaPredictTotal = miaAttacker.predict(attackerTotalScaled)
    miaPredictTrain = miaAttacker.predict(attackerTrainScaled)
    miaPredictTest = miaAttacker.predict(attackerTestScaled)

    totalAcc = round(metrics.accuracy_score(attackerLabels, miaPredictTotal), 4)
    trainAcc = round(metrics.accuracy_score(numpy.ones(len(predTrain)), miaPredictTrain), 4)
    testAcc = round(metrics.accuracy_score(numpy.zeros(len(predTest)), miaPredictTest), 4)

    totalFalse = numpy.count_nonzero(miaPredictTotal == 0)
    totalTrue = numpy.count_nonzero(miaPredictTotal == 1)

    print("MIA Model Training Complete!")
    return miaAttacker, totalAcc, trainAcc, testAcc, totalFalse, totalTrue   

print("Training Intitial Model...", end='\r')
randomStan = random.randrange(2048)

news = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'), random_state=randomStan)

trainData, testData, trainTarget, testTarget = train_test_split(
        news.data, news.target, train_size=11314, stratify=news.target, random_state=42
                                                                )

#Ignore any words within stopWords
stopWords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now"]
#Maximum number of features/words considered in the Model
maxFeatures = 20000
vectoriserStan = TIV(stop_words=stopWords, max_features=maxFeatures)
vectorsTrain = vectoriserStan.fit_transform(trainData)
vectorsTest = vectoriserStan.transform(testData)

#For Retraining
classifierInit = MultinomialNB(alpha=0.1)

classifier = MultinomialNB(alpha=0.1)
classifier.fit(vectorsTrain, trainTarget)
predTrain = classifier.predict(vectorsTrain)
predTest = classifier.predict(vectorsTest)
print("Initial Model Successfully Trained!")

#Get the Accuracy of the Model on the data sets
avgTrain = round(metrics.accuracy_score(trainTarget, predTrain), 4)
avgTest = round(metrics.accuracy_score(testTarget, predTest), 4)
trainingSetSize = len(trainData)

#Get the Log Loss on the data sets
lLossTrain = round(logLoss(trainTarget, classifier.predict_proba(vectorsTrain)), 4)
lLossTest = round(logLoss(testTarget, classifier.predict_proba(vectorsTest)), 4)

#Generate MIA model
#1, 4, 8, 16, 32, 64, 128, 256
scaler = StandardScaler()
mia, miaAccTotal, miaAccTrain, miaAccTest, miaBF, miaBT = miaBuilder(4, classifier.predict_log_proba(vectorsTrain), classifier.predict_log_proba(vectorsTest))
print("")

#Make list of all Classifiers
allClassifier = [classifier]
allNames = ["Initial Model"]

In [ ]:
#Introduction to Machine Learning and Unlearning
introExplain = mr.Expander(label="Introduction")

#Expander Text
with introExplain:
    display(Markdown('''**What is Machine Learning?**\n
Machine learning can be defined as a way of recognising patterns within a set of input (training) data, which can then be used for decision making purposes\n

**How does this Machine Learning Model Work?**\n
In this application, we use the 20Newsgroups data set. This data set contains approximately 18,000 posts gathered from one of the 20 following Newsgroups based on given topics:\n
-alt.atheism\n
-comp.graphics\n
-comp.os.ms-windows.misc\n
-comp.sys.ibm.pc.hardware\n
-comp.windows.x\n
-misc.forsale\n
-rec.autos\n
-rec.motorcycles\n
-rec.sport.baseball\n
-rec.sport.hockey\n
-sci.crypt\n
-sci.electronics\n
-sci.med\n
-sci.space\n
-soc.religion.christian\n
-talk.politics.guns\n
-talk.politics.mideast\n
-talk.politics.misc\n
-talk.religion.misc\n
The role of the machine learning models will be to correctly identify the topic of a given post based off of the text of 
the post itself, excluding the header, footer, and any other posts the post may be quoting.\n
To do this, the program transforms the text data into a list of TF-IDF values for each document, which is the product of term's frequency (TF) and 
the term's inverse document frequency (IDF), which represents how many documents use the term. By training on this data, being provided a documents TF-IDF 
values as well as it's topic, the model can find links between a document's TF-IDF values and it's topic, allowing the topic to be predicted.\n

**What is Machine Unlearning?**\n
Machine unlearning is the process of removing the information a model has learned from a given set of data, the forget set, that it had originally been trained on.\n
This may be done for many reasons, but the most common reasons are due to improper or poisoned data, or due to a Right to be Forgotten request by a person involved in the data.\n
In this application, there are several different unlearning methods to inspect, and their descriptions can be found in the "Input Values" dropdown in the sidebar.\n

**What is a Data Residual?**\n
A residual can be defined as the amount of impact forgotten data has on a model after the unlearning process. 
The simplest form of residuals can be seen in unlearning methods that don't retrain the model, but instead continue to fit 
the model in a way to unlearn the data to forget, such as Fine-Tune, NegGrad, and NegGrad+, but this residual can also be seen 
how the model itself is different compared to a fully retrained model that never learned information from the forget set, 
such as through the Weight Distance.\n
While pointing out the impact of the residuals can be difficult, showing the difference between the model that has undergone the 
forgetting method and a retrained model that never used the data to be forgotten allows the differences and residuals through 
the structure of the models to be clearly shown.
'''))

#MIA Section
display(Markdown("## Mia Accuracy:"))
#Expander to explain MIA
miaDetails = mr.Expander(label="MIA")

#Expander Text
with miaDetails:
    display(Markdown('''**What is MIA?**\n
MIA, or Membership Inference Attack, is a model trained to determine the likelihood that another model has been trained on a set of data.\n
If a model has been trained on a given data point, it is called a 'member' of the training data set, otherwise, it is called a 'non-member'.\n
It does this by comparing the data points with the class likelihoods provided by the model, comparing the model's confidence with 
training examples of the confidence other classification models had with similar data, and whether or not those models had the data 
point as a member or not.

**What can MIA tell us for Machine Unlearning?**\n
In simple terms, an optimal machine unlearning method should have lost any information it had previously learned from the forget data set. 
Therefore, the MIA model should be able to class the unlearned data as non-members for the classification model that underwent the forgetting 
process.\n
In other words, the greater the MIA's accuracy on the forget set for a given model, the greater degree that the information has been unlearned.\n
However, since the MIA model is not absolutely accurate, it's important to compare a model's MIA score with the MIA score of a Retrained model, 
as well as the initial accuracy of the MIA model on the non-members set.
                     
**Assessing Residuals from the Forget Set**\n
The residuals of the forget set is the impact that the data has had on the machine learning model after the unlearning process has been 
attempted. As we can't look at the exact, definite impact that every data point has had, this problem doesn't have a simple, catch-all solution. 
Instead, the different metrics of the different models must be consulted.\n
Bases: The best way to assess residuals is through the comparison of models that you already know the residuals of. As the intial model has 
been trained on the forget set, and has had no unlearning method done, it will be expected that it's decision making process will be influenced 
through the learned information of the forget set the most. Conversely, the retraining models will always be trained entirely independently 
of the forget set, meaning that it should have no residuals from the forget set. Every other method should then lie in between these two 
models' metrics.\n
Metrics: Now that there are references to compare methods to, the metrics can now be used as points of comparison. The key areas for residuals are:\n
Accuracy and Log Loss on the Forget Set compared to the Test Set: While minimal residuals, these two values should be near identical.\n
Accuracy and Log Loss on the Retain Set and Test Set from the Retrained model: If these values are substantially different to those in another 
model, it suggests that catastrophic loss has occured on the general accuracy of the model, which may be considered residuals that have 
negatively effected the model as a whole.\n
MIA Forget and the Forget Rate: The forget accuracy should resemble the untouched MIA model's accuracy on the non-member data. Any certainty 
above would suggest catastrophic loss, and certainty below suggests intial information learned from the forget set has remained. The forget 
rate also helps to reflect this, showing the degree of which the important information learned from the forget set has been retroactively 
forgotten.\n
Weight Distance: By comparing Weight Distances between a model and the related retrained model, the impact of the unlearning process that might 
otherwise not be reflected by other metrics can become apparent. By having differing weight distances, this means that the decision making 
processes of the models differ, meaning that while the overall accuracy and log loss may be similar, the exact samples that are correctly and 
incorrectly classified may differ greatly. This also means that any future changes made to both models may have different impacts, meaning a 
potential form of residuals that would manifest later on.
'''))

#MIA Accuracy + Model Metrics Header
display(Markdown('''Members Set: {}\n
Non-members Set: {}\n
## Model Metrics:'''.format(miaAccTrain, miaAccTest)))

#Expander to explain the Table Fields
tableDetails = mr.Expander(label="Table Details")

#Expander Text
with tableDetails:
    display(Markdown('''**Model**\n
This field gives the name of the unlearning method, as well as the Input Values used for the unlearning. Below is a description for the annotation used:\n
Unlearning Methods: RT (Retrained), F-T (Fine-Tune), NG (NegGrad), NGP (NegGrad+), BT (BadTeacher)\n
Forget Data Set Size: C-0.1 (Cluster-0.1), C-1 (Cluster-1), C-10 (Cluster-10), K (keith@cco.caltech.edu), M (mmm@cup.portal.com), and D (denning@guvax.acc.georgetown.edu)\n
Epochs: E:x (Maximum of x epochs allowed)\n
The two exceptions here are "Initial Model", which of course is done independently of the Input Values, and "BT: Initial", which is 
the original model for BadTeacher before the unlearning process. For more information, please read it's 
overview in the "Input Values" dropdown in the sidebar.\n
                     
**|Training Set|**\n
This field represents the size of the data set used for the unlearning method. For the unlearning methods in this application, 
the size will be the retain set, the forget set, or both sets combined.\n

**Accuracy and Log Loss: Retain, Test, and Forget**\n
By giving the accuracy of all three subsets comprising the 20Newsgroups data set, we can see some trade-offs between unlearning the forget 
set and accuracy loss on the training and test data sets respectively.\n
Additionally, the accuracy on the test and forget data sets can be compared. Ideally, these values should be as similar as possible, 
representing that the data to be forgotten has been unlearned to the extent that it is equivalent to unseen test data.\n
The multi-class log loss is also used to represent how confident the model is, as the metric compares the log of each 
probability of every class for every target, meaning large incorrect probabilities and small correct probabilities leads to a worse, 
larger log loss\n

**MIA: Forget and Forget Rate**\n
The MIA accuracy on the forget data set represents the certainty of the MIA model that the data set has not been used for training the relevant model.\n
The forget rate then builds on this, giving a percentage of the data points within the forget set that were originally classified as members 
of the model that are now classed as non-members of the model, giving a rate at which the model has unlearned from important data points to 
its decision making process.\n
For the MIA, it is important to consider it's original accuracy for non-members, which means that the values in this field 
should never be substantially higher than the ideal value given.\n

**Weight Distance**\n
This field represents the average change in model weights from the initial model to the model of the record. A larger value means that the 
model has been substantially changed by the unlearning process, which can be seen as unideal as a potential form of residuals.\n
By having a larger weight distance, this means that any future changes to the model will likely not reflect how the original model would 
be changed, making the unlearning process more noticeable as the new model would now deviate from what would be expected from the initial model.\n'''))

In [ ]:
#Intitial table data
tableData = pd.DataFrame({
    "Model": ["Initial Model"],
    "|Training Set|": [trainingSetSize],
    "Acc., LL.: Retain": [str(avgTrain) + ", " + str(lLossTrain)],
    "Acc., LL.: Test": [str(avgTest) + ", " + str(lLossTest)],
    "Acc., LL.: Forget": ["-"],
    "MIA: Forget": ["-"],
    "Forget Rate": ["-"],
    "Weight Distance": ["-"],
})

#Table output and adjustable hyperparameters
unlearnMethInput = mr.MultiSelect(
    label="Unlearning Methods",
    choices=["Fine-Tune", "NegGrad", "NegGrad+", "BadTeacher"]
)
toForgetInput = mr.Select(
    label="Forget Data Set",
    choices=["Cluster-0.1", "Cluster-1", "Cluster-10", "keith@cco.caltech.edu", "mmm@cup.portal.com", "denning@guvax.acc.georgetown.edu"]
)
maxEpochsInput = mr.NumberInput(
    label="Epochs per Unlearning Method",
    min=1,
    max=50,
    step=1
)

#Start unlearning process
unlearnBtn = mr.Button(
    label="Start",
    position="sidebar"
)

settingDetails = mr.Expander(label="Input Values")

with settingDetails:
    display(Markdown('''**Input Values**\n
These variables will change parts of the unlearning process. Below is an explanation of each category and the impact of their choices\n
                     
**Unlearning Methods**\n
The functions that will be used for unlearning the forget set. When selecting multiple unlearning methods, they will be used on separate models.\n
Retraining: The model is fully retrained from scratch, with the retain set as the training set. Due to the ideal results of the model, 
excluding time/energy cost, this method is always one of the methods done by the program.\n
Fine-Tune: The model continues to be trained on the retain set. While this is a simple and fast approach, the forget data set is never 
truly forgotten, just made less integral to the model's decision making process.\n
NegGrad: The model continues to be trained on the forget set. However, the TF-IDF values of the training data is inverted, meaning that the 
results of the training process is also inverted, and the model is made to unlearn the original information and perform worse on the forget set. 
However, this loss can impact the model's accuracy on the retain and test data sets due to the improper training process, and if 
the method is done in excess, the accuracy on the forget set will be significantly less than the accuracy on the test set, failing the 
original aim of making the forget set 'forgotten' and indistinguishable from the test set.\n
NegGrad+: This method combines the impact of both the Fine-Tune and NegGrad unlearning methods in order to gain both of the benefits. 
By training with both the training set and the inverse values of the forget set, the model can quickly reach satisfactory levels of 
unlearning without a focus being taken away from the information gained by the training set.\n
BadTeacher: This method combines the key ideas of retraining and fine-tuning by having two 'teacher' models, a good teacher and a bad teacher. 
Initially, the good teacher is trained as normal, and the bad teacher is trained on random information, making it equivalent to randomly 
guessing. Then, a student is designed by training on the retain set with labels given by the good teacher, before then training on the forget 
set with labels given by the bad teacher, and whenever a new forget request is made, a new student model is trained by the good teacher and 
fine tuned by the bad teacher to minimise any information gained by learning from the good teacher that has been trained on all forget set data.\n
                     
**Forget Data Set**\n
This determines what information within the training set will be the set of information to unlearn, called the forget set. Training data that 
is not also within the forget set is called the retain set.\n
Cluster-x: A cluster is a random assortment of data points from within the training set. The value of x corresponds to the percentage of the 
training set that is used in the cluster, so a larger value of x means a larger forget set and a smaller retain set. This may be used when a 
model's training data is believed to be poisoned, but without any identifying features, where they can attempt to remove information from the 
data set in an attempt to see which data portions are negatively effecting the model's metrics.\n
Emails: This field also provides three different emails, which may be used to reflect what may happen in the event of a right to be forgotten 
request. These choices remove any fields that mention the user's email, which can lead to class-specific accuracy loss, rather than the more 
general loss of clustering, which may lead to a more drastic loss in average accuracy as the retain set becomes uneven in terms of class 
representation.\n
            
**Epochs**\n
Epochs are the number of iterations that a model is trained over the same training set. By increasing this value, models based around fine-tuning 
adjustments (Fine-Tune, NegGrad, NegGrad+, and BadTeacher) will have a greater reduction in accuracy in the forget set as the unlearning process is 
increased. However, as mentioned, this can also lead to catastrophic loss on the accuracy for the retain and test data sets, so choosing the 
right amount of epochs becomes more of a decision of such trade-off.'''))

In [ ]:
#Intitial table to add data to later
tbl = mr.Table(tableData)

def addToTable(name, retainSize, avgTrain, avgTest, avgForget, miaForget, afterFalse, wd, lLossTrain, lLossTest, lLossForget):
    newTable = pd.DataFrame({
        "Model": [name],
        "|Training Set|": [retainSize],
        "Acc., LL.: Retain": [str(avgTrain) + ", " + str(lLossTrain)],
        "Acc., LL.: Test": [str(avgTest) + ", " + str(lLossTest)],
        "Acc., LL.: Forget": [str(avgForget) + ", " + str(lLossForget)],
        "MIA: Forget": [miaForget],
        "Forget Rate": [str(round(((afterFalse - beforeFalse) / beforeTrue) * 100, 2)) + "%"],
        "Weight Distance": [wd],
    })

    global tableData
    tableData = pd.concat([tableData, newTable])
    tbl.data = tableData

def addToFigure(values, name):
    figLine, = ax.plot(values)
    figLine.set_label(name)
    ax.legend()

def getModelName(method, set, epochs):
    name = ""
    #Get method abbreviation
    if method == "Retraining":
        name += "RT |"
    elif method == "Fine-Tune":
        name += "F-T |"
    elif method == "NegGrad":
        name += "NG |"
    elif method == "NegGrad+":
        name += "NGP |"
    else:
        name += "BT |"
    
    #Get forget set abbreviation
    if set == "Cluster-0.1":
        name += " C-0.1 |"
    elif set == "Cluster-1":
        name += " C-1 |"
    elif set == "Cluster-10":
        name += " C-10 |"
    elif set == "keith@cco.caltech.edu":
        name += " K |"
    elif set == "mmm@cup.portal.com":
        name += " M |"
    else:
        name += " D |"
    
    #Add max epochs
    name += " E:" + str(epochs)

    return name


def getForgetSize():
    if toForgetInput.value == "Cluster-0.1":
        return 0.1
    elif toForgetInput.value == "Cluster-1":
        return 1
    else:
        return 10

def getForgetRetainSets(newsTrainData, newsTrainTarget):
    if toForgetInput.value in ["keith@cco.caltech.edu", "mmm@cup.portal.com", "denning@guvax.acc.georgetown.edu"]:
        counts = getMatches(toForgetInput.value)
        retain = newsTrainData.copy()
        retainTarget = newsTrainTarget
        forget = []
        forgetTarget = []
        for i in reversed(counts):
            forget.append(retain[i])
            forgetTarget.append(retainTarget[i])
            del retain[i]
            #retain.pop(i)
            retainTarget = numpy.delete(retainTarget, i)
        return forget, retain, forgetTarget, retainTarget
    else:
        size = getForgetSize() / 100
        forget, retain, forgetTarget, retainTarget = train_test_split(newsTrainData, newsTrainTarget, train_size=size)
        return forget, retain, forgetTarget, retainTarget

def getWeightDistance(model1, model2):
    weightTotal = 0
    for i in range(20):
        for j in range(maxFeatures):
            weightTotal = weightTotal + (model1.feature_log_prob_[i,j] - model2.feature_log_prob_[i,j])**2
    weightTotal = (weightTotal ** 0.5) / (20*maxFeatures)
    return weightTotal


def getMIABFBT(mia, classifier, forget):
    forgetVectors = vectoriserStan.transform(forget)
    forgetPred = classifier.predict_log_proba(forgetVectors)
    global scaler
    forgetPredScaled = scaler.transform(forgetPred)

    miaPred = mia.predict(forgetPredScaled)

    return numpy.count_nonzero(miaPred == 0), numpy.count_nonzero(miaPred == 1)

def getMIAScore(mia, model, vectorsRetain, vectorsForget):
    #Get predictions probabilities for all 20 classes
    miaRetain = model.predict_log_proba(vectorsRetain)
    miaForget = model.predict_log_proba(vectorsForget)

    #Ones: Trained, Zeros: Not trained
    miaForgetVals = numpy.zeros(len(miaForget))

    #Scale the outputs to standardise the data
    scaler = StandardScaler()
    scaler.fit(numpy.concatenate((miaRetain, miaForget)))
    miaForgetScaled = scaler.transform(miaForget)

    #Get predictions of members and non-members
    forgetPred = mia.predict(miaForgetScaled)

    #Get accuracy scores
    accMiaForget = round(metrics.accuracy_score(miaForgetVals, forgetPred), 4)

    #Get AF
    afterFalse = numpy.count_nonzero(forgetPred == 0)

    return accMiaForget, afterFalse

def retraining(train, test, forget, epochs):
    #Each parameter is a 2-tuple of the relevant data and the relevant targets
    #Build Retrained model
    #Keep using work done on the test data set from the original model
    vectorsTrain = vectoriserStan.transform(train[0])
    vectorsTest = vectoriserStan.transform(test[0])
    vectorsForget = vectoriserStan.transform(forget[0])

    #Train model on retain set
    rtrClassifier = deepcopy(classifierInit)
    rtrClassifier.fit(vectorsTrain, train[1])

    for i in range(epochs):
        rtrClassifier.partial_fit(vectorsTrain, train[1])

    #Test new model on test and forget sets
    predForget = rtrClassifier.predict(vectorsForget)
    predTest = rtrClassifier.predict(vectorsTest)
    predRetain = rtrClassifier.predict(vectorsTrain)

    #Get Accuracy on the three data sets
    avgTest = round(metrics.accuracy_score(test[1], predTest), 4)
    avgForget = round(metrics.accuracy_score(forget[1], predForget), 4)
    avgRetain = round(metrics.accuracy_score(train[1], predRetain), 4)

    #Get Multi-class Log Loss on the three data sets
    lLossTest = round(logLoss(test[1], rtrClassifier.predict_proba(vectorsTest)), 4)
    lLossForget = round(logLoss(forget[1], rtrClassifier.predict_proba(vectorsForget)), 4)
    lLossRetain = round(logLoss(train[1], rtrClassifier.predict_proba(vectorsTrain)), 4)

    #Get MIA on the forget set
    miaForget, afterFalse = getMIAScore(mia, rtrClassifier, vectorsTrain, vectorsForget)

    #Get Weight Distance metric
    wd = round(getWeightDistance(classifier, rtrClassifier), 4)

    return avgRetain, avgTest, avgForget, miaForget, afterFalse, wd, (lLossRetain, lLossTest, lLossForget), rtrClassifier

def fineTuning(train, test, forget, epochs):
    #Build off the Initial Model
    classifierFT = deepcopy(classifier)

    #Vectorise values
    vectorsTrain = vectoriserStan.transform(train[0])
    vectorsTest = vectoriserStan.transform(test[0])
    vectorsForget = vectoriserStan.transform(forget[0])

    totalAfterFalses = []
    curr = 0
    for i in range(epochs):
        _, currAfterFalse = getMIAScore(mia, classifierFT, vectorsTrain, vectorsForget)
        totalAfterFalses.append(round(((currAfterFalse - beforeFalse) / beforeTrue) * 100, 2))
        curr += 1
        classifierFT.partial_fit(vectorsTrain, train[1], classes=numpy.unique(train[1]))

    #Get Predictions on the three data sets
    predTrain = classifierFT.predict(vectorsTrain)
    predTest = classifierFT.predict(vectorsTest)
    predForget = classifierFT.predict(vectorsForget)

    #Get Accuracy on the three data sets
    avgTrain = round(metrics.accuracy_score(train[1], predTrain), 4)
    avgTest = round(metrics.accuracy_score(test[1], predTest), 4)
    avgForget = round(metrics.accuracy_score(forget[1], predForget), 4)

    #Get Multi-class Log Loss on the three data sets
    lLossTest = round(logLoss(test[1], classifierFT.predict_proba(vectorsTest)), 4)
    lLossForget = round(logLoss(forget[1], classifierFT.predict_proba(vectorsForget)), 4)
    lLossTrain = round(logLoss(train[1], classifierFT.predict_proba(vectorsTrain)), 4)

    #Get MIA on the forget set
    miaForget, afterFalse = getMIAScore(mia, classifierFT, vectorsTrain, vectorsForget)
    totalAfterFalses.append(round(((afterFalse - beforeFalse) / beforeTrue) * 100, 2))

    #Get Weight Distance metric
    wd = round(getWeightDistance(classifier, classifierFT), 4)

    return avgTrain, avgTest, avgForget, miaForget, afterFalse, wd, (lLossTrain, lLossTest, lLossForget), classifierFT, totalAfterFalses

def negGrad(train, test, forget, epochs):
    #Build off the Initial Model
    classifierNG = deepcopy(classifier)
    #Vectorise values
    vectorsTrain = vectoriserStan.transform(train[0])
    vectorsTest = vectoriserStan.transform(test[0])
    vectorsForget = vectoriserStan.transform(forget[0])
    vectorsGradAsc = vectorsForget

    #Invert values for gradient ascent
    vectorsGradAsc = numpy.asarray(1 - (vectorsGradAsc.todense()))

    totalAfterFalses = []
    curr = 0

    #Gradient Ascent on the (Negated) Forget Set
    for i in range(epochs):
        _, currAfterFalse = getMIAScore(mia, classifierNG, vectorsTrain, vectorsForget)
        totalAfterFalses.append(round(((currAfterFalse - beforeFalse) / beforeTrue) * 100, 2))
        curr += 1
        classifierNG.partial_fit(vectorsGradAsc, forget[1], classes=numpy.unique(train[1]))
    
    #Get Predictions on the three data sets
    predTrain = classifierNG.predict(vectorsTrain)
    predTest = classifierNG.predict(vectorsTest)
    predForget = classifierNG.predict(vectorsForget)

    #Get Accuracy on the three data sets
    avgTrain = round(metrics.accuracy_score(train[1], predTrain), 4)
    avgTest = round(metrics.accuracy_score(test[1], predTest), 4)
    avgForget = round(metrics.accuracy_score(forget[1], predForget), 4)

    #Get Multi-class Log Loss on the three data sets
    lLossTest = round(logLoss(test[1], classifierNG.predict_proba(vectorsTest)), 4)
    lLossForget = round(logLoss(forget[1], classifierNG.predict_proba(vectorsForget)), 4)
    lLossTrain = round(logLoss(train[1], classifierNG.predict_proba(vectorsTrain)), 4)

    #Get MIA on the forget set
    miaForget, afterFalse = getMIAScore(mia, classifierNG, vectorsTrain, vectorsForget)
    totalAfterFalses.append(round(((afterFalse - beforeFalse) / beforeTrue) * 100, 2))

    #Get Weight Distance metric
    wd = round(getWeightDistance(classifier, classifierNG), 4)

    return avgTrain, avgTest, avgForget, miaForget, afterFalse, wd, (lLossTrain, lLossTest, lLossForget), classifierNG, totalAfterFalses

def negGradPlus(train, test, forget, epochs, ratio):
    #Build off the Initial Model
    classifierNGP = deepcopy(classifier)

    #Vectorise values
    vectorsTrain = vectoriserStan.transform(train[0])
    vectorsTest = vectoriserStan.transform(test[0])
    vectorsForget = vectoriserStan.transform(forget[0])
    vectorsGradAsc = vectorsForget

    #Invert values for gradient ascent
    vectorsGradAsc = numpy.asarray(1 - (vectorsGradAsc.todense()))

    totalAfterFalses = []
    curr = 0

    for i in range(epochs):
        _, currAfterFalse = getMIAScore(mia, classifierNGP, vectorsTrain, vectorsForget)
        totalAfterFalses.append(round(((currAfterFalse - beforeFalse) / beforeTrue) * 100, 2))
        curr += 1
        #Ratio = 1: Fine-Tune, Ratio = 0: NegGrad
        #Fine-Tune
        classifierNGP.partial_fit(vectorsTrain, train[1], classes=numpy.unique(train[1]), sample_weight=ratio)

        #NegGrad
        classifierNGP.partial_fit(vectorsGradAsc, forget[1], classes=numpy.unique(train[1]), sample_weight=1-ratio)
    
    #Get Predictions on the three data sets
    predTrain = classifierNGP.predict(vectorsTrain)
    predTest = classifierNGP.predict(vectorsTest)
    predForget = classifierNGP.predict(vectorsForget)

    #Get Accuracy on the three data sets
    avgTrain = round(metrics.accuracy_score(train[1], predTrain), 4)
    avgTest = round(metrics.accuracy_score(test[1], predTest), 4)
    avgForget = round(metrics.accuracy_score(forget[1], predForget), 4)

    #Get Multi-class Log Loss on the three data sets
    lLossTest = round(logLoss(test[1], classifierNGP.predict_proba(vectorsTest)), 4)
    lLossForget = round(logLoss(forget[1], classifierNGP.predict_proba(vectorsForget)), 4)
    lLossTrain = round(logLoss(train[1], classifierNGP.predict_proba(vectorsTrain)), 4)

    #Get MIA on the forget set
    miaForget, afterFalse = getMIAScore(mia, classifierNGP, vectorsTrain, vectorsForget)
    totalAfterFalses.append(round(((afterFalse - beforeFalse) / beforeTrue) * 100, 2))

    #Get Weight Distance metric
    wd = round(getWeightDistance(classifier, classifierNGP), 4)

    return avgTrain, avgTest, avgForget, miaForget, afterFalse, wd, (lLossTrain, lLossTest, lLossForget), classifierNGP, totalAfterFalses

def badTeacher(train, test, forget, epochs):
    #Good Teacher can be Initial Model
    goodClassifier = deepcopy(classifier)
    #Bad Teacher must be trained on Retain Set with random outputs
    badClassifier = MultinomialNB(alpha=0.1)

    #Vectorise values
    vectorsTrain = vectoriserStan.transform(train[0])
    vectorsTest = vectoriserStan.transform(test[0])
    vectorsForget = vectoriserStan.transform(forget[0])

    #Randomise the TF-IDF values of features for badClassifier
    #CountVectorizer and TfidfTransformer combined is equivalent to TIV
    cv = CountVectorizer(stop_words=stopWords, max_features=maxFeatures)
    trainCounts = cv.fit_transform(train[0])
    forgetCounts = cv.transform(forget[0])
    #Shuffle rows of forgetCounts
    numpy.random.shuffle(trainCounts.todense())
    numpy.random.shuffle(forgetCounts.todense())
    tfidf = TfidfTransformer()
    vectorsTrainBad = tfidf.fit_transform(trainCounts)
    vectorsBad = tfidf.transform(forgetCounts)

    #Train badClassifier on the Training and Forget data sets
    badClassifier.fit(vectorsBad, [random.randint(0,20) for _ in range(len(forget[1]))])

    #Build Student Model
    studentClassifier = MultinomialNB(alpha=0.1)

    #Train based off goodClassifier predictions on the Training data set
    studentClassifier.fit(vectorsTrain, goodClassifier.predict(vectorsTrain))

    #Get Predictions on the three data sets BEFORE training from badClassifier
    predTrainBef = studentClassifier.predict(vectorsTrain)
    predTestBef = studentClassifier.predict(vectorsTest)
    predForgetBef = studentClassifier.predict(vectorsForget)

    #Get Accuracy on the three data sets
    avgTrainBef = round(metrics.accuracy_score(train[1], predTrainBef), 4)
    avgTestBef = round(metrics.accuracy_score(test[1], predTestBef), 4)
    avgForgetBef = round(metrics.accuracy_score(forget[1], predForgetBef), 4)

    #Get Multi-class Log Loss on the three data sets
    lLossTestBef = round(logLoss(test[1], studentClassifier.predict_proba(vectorsTest)), 4)
    lLossForgetBef = round(logLoss(forget[1], studentClassifier.predict_proba(vectorsForget)), 4)
    lLossTrainBef = round(logLoss(train[1], studentClassifier.predict_proba(vectorsTrain)), 4)

    #Get MIA on the forget set
    miaForgetBef, afterFalseBef = getMIAScore(mia, studentClassifier, vectorsTrain, vectorsForget)

    #Get Weight Distance metric
    wdBef = round(getWeightDistance(classifier, studentClassifier), 4)

    totalAfterFalses = []
    curr = 0

    for i in range(epochs):
        _, currAfterFalse = getMIAScore(mia, studentClassifier, vectorsTrain, vectorsForget)
        totalAfterFalses.append(round(((currAfterFalse - beforeFalse) / beforeTrue) * 100, 2))
        curr += 1
        #Train based off badClassifier predictions on the Forget data set
        studentClassifier.partial_fit(vectorsForget, badClassifier.predict(vectorsForget))
        studentClassifier.partial_fit(vectorsTrain, goodClassifier.predict(vectorsTrain))

    #Get Predictions on the three data sets AFTER training from badClassifier
    predTrainAft = studentClassifier.predict(vectorsTrain)
    predTestAft = studentClassifier.predict(vectorsTest)
    predForgetAft = studentClassifier.predict(vectorsForget)

    #Get Accuracy on the three data sets
    avgTrainAft = round(metrics.accuracy_score(train[1], predTrainAft), 4)
    avgTestAft = round(metrics.accuracy_score(test[1], predTestAft), 4)
    avgForgetAft = round(metrics.accuracy_score(forget[1], predForgetAft), 4)

    #Get Multi-class Log Loss on the three data sets
    lLossTestAft = round(logLoss(test[1], studentClassifier.predict_proba(vectorsTest)), 4)
    lLossForgetAft = round(logLoss(forget[1], studentClassifier.predict_proba(vectorsForget)), 4)
    lLossTrainAft = round(logLoss(train[1], studentClassifier.predict_proba(vectorsTrain)), 4)

    #Get MIA on the forget set
    miaForgetAft, afterFalseAft = getMIAScore(mia, studentClassifier, vectorsTrain, vectorsForget)
    totalAfterFalses.append(round(((afterFalseAft - beforeFalse) / beforeTrue) * 100, 2))

    #Get Weight Distance metric
    wdAft = round(getWeightDistance(classifier, studentClassifier), 4)

    return (
        avgTrainBef, avgTestBef, avgForgetBef, miaForgetBef, afterFalseBef, wdBef, (lLossTrainBef, lLossTestBef, lLossForgetBef)
        ), (
        avgTrainAft, avgTestAft, avgForgetAft, miaForgetAft, afterFalseAft, wdAft, (lLossTrainAft, lLossTestAft, lLossForgetAft)
        ), studentClassifier, totalAfterFalses

In [ ]:
#Activate when unlearnBtn is clicked
if unlearnBtn.value:
    #Initial Figure
    plt.pause(0.1)
    fig, ax = plt.subplots()
    plt.ylim(0, 100)
    plt.xlim(0, 50)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_title("20Newsgroups Unlearning Methods")
    ax.set_ylabel("Forget Rate")
    ax.set_xlabel("Model Iteration")

    #Get unlearning hyperparameters
    unlearnMethods = unlearnMethInput.value
    forgetSet, retainSet, forgetTarget, retainTarget = getForgetRetainSets(trainData, trainTarget)
    maxEpochs = int(maxEpochsInput.value)

    #Get Before False and Before True MIA values for the forget set
    beforeFalse, beforeTrue = getMIABFBT(mia, classifier, forgetSet)

    #Get Retrained model
    rtrAvgTrain, rtrAvgTest, rtrAvgForget, rtrMiaForget, rtrAfterFalse, rtrWD, (rtrLLTrain, rtrLLTest, rtrLLForget), rtrRetraining = retraining(
        [retainSet, retainTarget], [testData, testTarget], [forgetSet, forgetTarget], maxEpochs
        )
    rtrName = getModelName("Retraining", toForgetInput.value, maxEpochs)
    allClassifier.append(deepcopy(rtrRetraining))
    allNames.append(rtrName)
    addToTable(rtrName, len(retainSet), rtrAvgTrain, rtrAvgTest, rtrAvgForget, rtrMiaForget, rtrAfterFalse, rtrWD, rtrLLTrain, rtrLLTest, rtrLLForget)
    addToFigure(numpy.full(maxEpochs+1, round(((rtrAfterFalse - beforeFalse) / beforeTrue) * 100, 2)), rtrName)

    #Get other unlearning methods
    for i in unlearnMethods:
        if i == "Fine-Tune":
            ftAvgTrain, ftAvgTest, ftAvgForget, ftMiaForget, ftAfterFalse, ftWD, (ftLLTrain, ftLLTest, ftLLForget), ftClassifier, ftTotalAfterFalses = fineTuning(
                [retainSet, retainTarget], [testData, testTarget], [forgetSet, forgetTarget], maxEpochs
                )
            ftName = getModelName("Fine-Tune", toForgetInput.value, maxEpochs)
            allClassifier.append(deepcopy(ftClassifier))
            allNames.append(ftName)
            addToTable(ftName, len(retainSet), ftAvgTrain, ftAvgTest, ftAvgForget, ftMiaForget, ftAfterFalse, ftWD, ftLLTrain, ftLLTest, ftLLForget)
            addToFigure(ftTotalAfterFalses, ftName)
        elif i == "NegGrad":
            ngAvgTrain, ngAvgTest, ngAvgForget, ngMiaForget, ngAfterFalse, ngWD, (ngLLTrain, ngLLTest, ngLLForget), ngClassifier, ngTotalAfterFalses = negGrad(
                [retainSet, retainTarget], [testData, testTarget], [forgetSet, forgetTarget], maxEpochs
                )
            ngName = getModelName("NegGrad", toForgetInput.value, maxEpochs)
            allClassifier.append(deepcopy(ngClassifier))
            allNames.append(ngName)
            addToTable(ngName, len(forgetSet), ngAvgTrain, ngAvgTest, ngAvgForget, ngMiaForget, ngAfterFalse, ngWD, ngLLTrain, ngLLTest, ngLLForget)
            addToFigure(ngTotalAfterFalses, ngName)
        elif i == "NegGrad+":
            ngpAvgTrain, ngpAvgTest, ngpAvgForget, ngpMiaForget, ngpAfterFalse, ngpWD, (ngpLLTrain, ngpLLTest, ngpLLForget), ngpClassifier, ngpTotalAfterFalses = negGradPlus(
                [retainSet, retainTarget], [testData, testTarget], [forgetSet, forgetTarget], maxEpochs, 0.995
                )
            ngpName = getModelName("NegGrad+", toForgetInput.value, maxEpochs)
            allClassifier.append(deepcopy(ngpClassifier))
            allNames.append(ngpName)
            addToTable(ngpName, len(retainSet)+len(forgetSet), ngpAvgTrain, ngpAvgTest, ngpAvgForget, ngpMiaForget, ngpAfterFalse, ngpWD, ngpLLTrain, ngpLLTest, ngpLLForget)
            addToFigure(ngpTotalAfterFalses, ngpName)
        elif i == "BadTeacher":
            (
                btAvgTrainBef, btAvgTestBef, btAvgForgetBef, btMiaForgetBef, btAfterFalseBef, btWDBef, (btLLTrainBef, btLLTestBef, btLLForgetBef)
            ), (
                btAvgTrainAft, btAvgTestAft, btAvgForgetAft, btMiaForgetAft, btAfterFalseAft, btWDAft, (btLLTrainAft, btLLTestAft, btLLForgetAft)
            ), btClassifier, btTotalAfterFalses = badTeacher(
                [retainSet, retainTarget], [testData, testTarget], [forgetSet, forgetTarget], maxEpochs
            )
            btName = getModelName("BadTeacher", toForgetInput.value, maxEpochs)
            allClassifier.append(deepcopy(btClassifier))
            allNames.append(btName)
            addToTable("BT: Original Model", len(retainSet)+len(forgetSet), btAvgTrainBef, btAvgTestBef, btAvgForgetBef, btMiaForgetBef, btAfterFalseBef, btWDBef, btLLTrainBef, btLLTestBef, btLLForgetBef)
            addToTable(btName, len(retainSet)+len(forgetSet), btAvgTrainAft, btAvgTestAft, btAvgForgetAft, btMiaForgetAft, btAfterFalseAft, btWDAft, btLLTrainAft, btLLTestAft, btLLForgetAft)
            addToFigure(btTotalAfterFalses, btName)
    plt.show()

In [ ]:
#Targets for Samples
def getTargetName(num):
    if num == 0:
        return "alt.atheism"
    elif num == 1:
        return "comp.graphics"
    elif num == 2:
        return "comp.os.ms-windows.misc"
    elif num == 3:
        return "comp.sys.ibm.pc.hardware"
    elif num == 4:
        return "comp.sys.mac.hardware"
    elif num == 5:
        return "comp.windows.x"
    elif num == 6:
        return "misc.forsale"
    elif num == 7:
        return "rec.autos"
    elif num == 8:
        return "rec.motorcycles"
    elif num == 9:
        return "rec.sport.baseball"
    elif num == 10:
        return "rec.sport.hockey"
    elif num == 11:
        return "sci.crypt"
    elif num == 12:
        return "sci.electronics"
    elif num == 13:
        return "sci.med"
    elif num == 14:
        return "sci.space"
    elif num == 15:
        return "soc.religion.christian"
    elif num == 16:
        return "talk.politics.guns"
    elif num == 17:
        return "talk.politics.mideast"
    elif num == 18:
        return "talk.politics.misc"
    elif num == 19:
        return "talk.religion.misc"

#Get random sample button
sampleBtn = mr.Button(
    label="Get Random Sample and Predictions",
    position="inline"
)

In [ ]:
#Sample Button Function
if sampleBtn.value:
    sampleData, sampleTest, sampleTestTarget = getExample()
    display(Markdown(sampleData))

    #Prepare sampleTest
    sampleTestVectors = vectoriserStan.transform([sampleTest])
    display(Markdown('**Actual class: {} ({})**'.format(sampleTestTarget, getTargetName(sampleTestTarget))))

    for i in range(len(allClassifier)):
        result = allClassifier[i].predict(sampleTestVectors)
        display(Markdown('{} predicted: {} ({})'.format(allNames[i], result, getTargetName(result))))